（No,6）

In [ ]:
import re
from pathlib import Path

# --- 共通の復元基盤 undo_utils.py を読み込む ---
import sys
from pathlib import Path


def _locate_undo_utils():
    """undo_utils.py があるフォルダを探す（VS Code の作業ディレクトリ設定に依存しない）。"""
    candidates = []
    # 1) VS Code がノートブック自身のパスを教えてくれる場合
    nb_file = globals().get("__vsc_ipynb_file__")
    if nb_file:
        candidates.append(Path(nb_file).parent)
    # 2) 作業ディレクトリと、その中／親の「コードフォルダ」
    cwd = Path.cwd()
    candidates += [cwd, cwd / "コードフォルダ", cwd.parent, cwd.parent / "コードフォルダ"]

    for c in candidates:
        if (c / "undo_utils.py").is_file():
            return c.resolve()

    raise FileNotFoundError(
        "undo_utils.py が見つかりません。\n"
        "このノートブックと同じ「コードフォルダ」内に undo_utils.py があるか確認してください。\n"
        f"探した場所: {[str(c) for c in candidates]}"
    )


_uu_dir = str(_locate_undo_utils())
if _uu_dir not in sys.path:
    sys.path.insert(0, _uu_dir)

import undo_utils as uu

選択したフォルダ内にある、三桁の連番ファイルのスタート番号を変えて連番数字を振り直す。
ただし、ファイル名は108_190℃_320_1.0mmの構成になっている必要がある。

In [ ]:
# === 三桁の連番ファイルのスタート番号を変えて連番を振り直す ===
# ファイル名は「108_190℃_320_1.0mm」のような「数字_残り」の構成である必要があります。
# 2段階リネーム（__temp_rename_ を経由）の各 rename を1件ずつ記録するので、
# 復元セルは逆順に再生するだけで正しく元の番号に戻せます。

# ==========================================
# 設定
# ==========================================
START_NUMBER = 1   # 新しい連番の開始番号
ZERO_PADDING = 3   # 連番の桁数（3 なら 001, 002, ...）
# ==========================================

STEP_NAME = "06_連番数字の振り直し"


def main():
    target_path = uu.select_folder("連番を振り直すフォルダを選択してください")
    if target_path is None:
        return

    # 行頭の数字と、アンダースコア以降の文字列を抽出
    # 例: "108_190℃_320_1.0mm.txt" -> group(1)="108", group(2)="_190℃_320_1.0mm.txt"
    pattern = re.compile(r"^(\d+)(_.*)$")

    # 処理対象ファイルを収集（このフォルダ直下のみ／_undo・_trash は除外）
    target_files = []
    for file_path in sorted(target_path.iterdir()):
        if not file_path.is_file():
            continue
        if uu.is_reserved(file_path, target_path):
            continue

        match = pattern.match(file_path.name)
        if match:
            target_files.append({
                "original_path": file_path,
                "original_number": int(match.group(1)),  # ソート用
                "suffix": match.group(2),
            })

    if not target_files:
        print("対象となるファイル（先頭が「数字＋アンダースコア」の形式）が見つかりませんでした。")
        return

    # 元の連番で昇順にソートし、順番を一切変えないようにする
    target_files.sort(key=lambda x: x["original_number"])

    # 既存のファイル名と新しいファイル名が衝突して上書きされるのを防ぐため、
    # 一旦すべて「一時的なファイル名」に変更してから目的の名前に変える2段階方式をとる。
    with uu.UndoJournal(target_path, STEP_NAME) as j:
        temp_files = []

        # 第1段階: 一時ファイル名にリネーム
        for i, file_data in enumerate(target_files):
            new_number_str = str(START_NUMBER + i).zfill(ZERO_PADDING)
            new_filename = f"{new_number_str}{file_data['suffix']}"

            temp_path = target_path / f"__temp_rename_{i}_{new_filename}"
            j.move(file_data["original_path"], temp_path)
            temp_files.append((temp_path, target_path / new_filename))

        # 第2段階: 最終的な名前にリネーム
        print(f"--- 選択フォルダ: {target_path} ---")
        for temp_path, final_path in temp_files:
            j.move(temp_path, final_path)
            print(f"変更完了: {final_path.name}")

    print()
    print(f"合計 {len(temp_files)} 個のファイル名の連番を振り直しました。")


main()

---
### ⏪ 復元（元に戻す）

このセルを実行すると、**このノートブックで行った直前の1工程**を巻き戻します。
（メインフォルダの `_undo/undo_log.json` に記録された履歴を使います）

繰り返し実行すれば、01〜06 のどの工程まででもさかのぼれます。
削除したファイルは `_trash` フォルダに退避されているので、これも一緒に元の場所へ戻ります。

In [ ]:
# ===== 共通の復元セル =====
# 直前に実行した1工程を巻き戻します。
# 続けて実行すれば、さらに1つ前の工程へとさかのぼれます。

uu.undo_interactive()